# Items 4 and 5: model size / inference time, and label-free test-time adaptation (AdaBN)

**No training happens in this notebook.** It only loads saved checkpoints and runs inference.

- **Section 7 (item 4):** parameter counts per component and inference time for the VQC and no-VQC models (random weights are enough for timing). About 5 minutes.
- **Section 8 (item 5):** for every saved VQC / no-VQC checkpoint, re-estimates the backbone's batch-norm statistics on *unlabeled* target images, then evaluates. No labels are used for the adaptation, and no weights change except the BN running means and variances.
  - APTOS-only models: targets IDRiD and DDR (deduplicated). Multi-source models: IDRiD only (DDR was in their training data).
  - Two protocols: **full** (adapt on all target images, evaluate on the same images; the usual AdaBN setting) and **held-out** (adapt on one part, evaluate on a part never seen: IDRiD official train 413 → test 103; DDR train 8,696 → test 1,864). Report held-out as the main result.
  - Every model is first evaluated *without* adaptation as a check: those numbers must match the ones already in your results files.

The three control families (classical-only, cnn_ms, cnn_single) are not included yet, because their model code is in `controls_batch.ipynb`. They will be added in a later version.

**Steps:** fill the two zip paths in cell 1, run all cells top to bottom with `SMOKE = True`, check the output, then set `SMOKE = False` and run from cell 2 down. Finished rows are skipped if the session breaks.

---
**Version 2 changes**
- Extra metrics: sensitivity at fixed specificity (0.90 and 0.80), per-class recall, referable-DR counts (TP/FN/TN/FP) and confusion matrices.
- APTOS test predictions are saved too (and, for multi-source models, DDR test as an internal held-out set), so all comparisons can be bootstrapped.
- Results go to a **new** file `results_tta_v2.csv` and predictions to the folder `tta_preds_v2/` (the old `results_tta.csv` has fewer columns, so it is left untouched).
- Summary now reports mean ± SD over seeds, the per-seed and mean ± SD AdaBN change, per-class recall and referable counts, all saved as CSV.
- New section 9: paired bootstrap intervals. New section 10: package versions for the paper.
- Section 7 (timing) is skipped by default because its results are already saved; set `RUN_TIMING = True` to repeat it.

## 1. Setup (same as the multi-source notebook)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile

APTOS_ZIP = "/content/drive/MyDrive/archive.zip"
IDRID_ZIP = "/content/drive/MyDrive/IDRID.zip"        # <-- CHANGE
DDR_ZIP   = "/content/drive/MyDrive/DDR.zip"          # <-- CHANGE

os.system("pip install -q timm pennylane pennylane-lightning")
if not os.path.exists("/content/APTOS"):
    zipfile.ZipFile(APTOS_ZIP).extractall("/content/APTOS")
if not os.path.exists("/content/idrid/IDRID"):
    os.system(f'unzip -q "{IDRID_ZIP}" -d /content/idrid')
if not os.path.exists("/content/ddr/DR_grading.csv"):
    os.system(f'unzip -q "{DDR_ZIP}" -d /content/ddr')
print(os.listdir("/content/APTOS"), os.listdir("/content/idrid"), os.listdir("/content/ddr"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['test.csv', 'train_images', 'valid.csv', 'test_images', 'val_images', 'train_1.csv'] ['IDRID'] ['DR_grading.csv', 'DR_grading']


## 2. Imports and settings

In [ ]:
import random, hashlib, time, re, glob
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import timm, cv2
import pennylane as qml
from concurrent.futures import ThreadPoolExecutor
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, roc_curve, confusion_matrix)
import logging, warnings, sys, importlib

# quieten the harmless timm 'Unexpected keys' and HF_TOKEN messages
logging.getLogger("timm").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

BASE_PATH = "/content/drive/MyDrive/your folder"

SMOKE = False     # True = first checkpoint only, IDRiD only. Set False for the full run.
RUN_TIMING = False   # section 7 results are already saved; set True only to repeat the timing

config = {"batch_size": 32}
print("SMOKE" if SMOKE else "FULL RUN")

FULL RUN


## 3. Augmentation definitions (only the validation transform is used here)

In [ ]:
train_tf = A.Compose([
    A.Resize(256, 256),
    A.RandomResizedCrop(size=(224, 224), scale=(0.85, 1.0)),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Rotate(limit=15),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

## 4. Data (identical to the multi-source notebook)

In [ ]:
def cache_one(a):
    s, d = a
    if not os.path.exists(d):
        im = cv2.imread(s)
        if im is None: raise RuntimeError("cannot read " + s)
        cv2.imwrite(d, cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA))
def cache_all(df):
    with ThreadPoolExecutor(4) as ex:
        list(tqdm(ex.map(cache_one, zip(df.src_path, df.cache)), total=len(df)))

APT_ROOT = "/content/APTOS"
ap_dir = {"train": f"{APT_ROOT}/train_images/train_images", "val": f"{APT_ROOT}/val_images/val_images",
          "test": f"{APT_ROOT}/test_images/test_images"}
ap = pd.read_csv(BASE_PATH + "/aptos_clean_split.csv")[["id_code", "diagnosis", "split"]].copy()
ap["source"] = "aptos"
ap["src_path"] = [f"{ap_dir[s]}/{i}.png" for s, i in zip(ap.split, ap.id_code)]
ap["cache"] = [f"/content/cache_aptos/{i}.png" for i in ap.id_code]

by_name = {}
for root, _, files in os.walk("/content/ddr"):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")): by_name.setdefault(f, []).append(os.path.join(root, f))
dd = pd.read_csv(BASE_PATH + "/ddr_clean_split.csv")[["id_code", "diagnosis", "split"]].copy()
dd["source"] = "ddr"
missing = [n for n in dd.id_code if n not in by_name]
assert not missing, f"{len(missing)} DDR images not found, e.g. {missing[:3]}"
dd["src_path"] = dd.id_code.map(lambda n: by_name[n][0])
dd["cache"] = [f"/content/cache_ddr/{i}.png" for i in dd.id_code]

alldf = pd.concat([ap, dd], ignore_index=True)
alldf["diagnosis"] = alldf["diagnosis"].astype(int)
for d in ("/content/cache_aptos", "/content/cache_ddr"): os.makedirs(d, exist_ok=True)
cache_all(alldf)
for (s, sp), gdf in alldf.groupby(["source", "split"]):
    print(f"{s:6s} {sp:5s} {len(gdf):6d}", np.bincount(gdf.diagnosis, minlength=5))

# IDRiD: all 516 graded images (used only in the final test of each run)
R = "/content/idrid/IDRID"
img_dirs  = {"train": f"{R}/1. Original Images/a. Training Set", "test": f"{R}/1. Original Images/b. Testing Set"}
lab_files = {"train": f"{R}/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv",
             "test":  f"{R}/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv"}
rows = []
for sub in ("train", "test"):
    lab = pd.read_csv(lab_files[sub]).iloc[:, :2].dropna(); lab.columns = ["name", "grade"]
    files = {os.path.splitext(f)[0]: os.path.join(img_dirs[sub], f) for f in os.listdir(img_dirs[sub])
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))}
    for _, r in lab.iterrows():
        n = str(r["name"]).strip()
        if n in files: rows.append(dict(sub=sub, name=n, src_path=files[n], diagnosis=int(r["grade"])))
idr = pd.DataFrame(rows)
idr["cache"] = [f"/content/cache_idrid/{s}_{n}.png" for s, n in zip(idr["sub"], idr["name"])]
os.makedirs("/content/cache_idrid", exist_ok=True); cache_all(idr)
print("IDRiD", len(idr), np.bincount(idr.diagnosis, minlength=5))

class CachedDS(Dataset):
    def __init__(self, df, tf):
        self.p, self.y, self.t = df.cache.tolist(), df.diagnosis.astype(int).tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = cv2.cvtColor(cv2.imread(self.p[i]), cv2.COLOR_BGR2RGB)
        return self.t(image=im)["image"], self.y[i]

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

bs = config["batch_size"]
train_df = alldf[alldf.split == "train"]
mk = lambda df: DataLoader(CachedDS(df, val_tf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
val_loaders  = {s: mk(alldf[(alldf.split == "val")  & (alldf.source == s)]) for s in ("aptos", "ddr")}
test_loaders = {s: mk(alldf[(alldf.split == "test") & (alldf.source == s)]) for s in ("aptos", "ddr")}
idrid_loader = mk(idr)
print("training images:", len(train_df))

100%|██████████| 15928/15928 [00:00<00:00, 124948.57it/s]


aptos  test     339 [197  28  75  12  27]
aptos  train   2824 [1427  277  756  145  219]
aptos  val      341 [172  33  91  20  25]
ddr    test    1864 [926  94 672  35 137]
ddr    train   8696 [4317  441 3134  165  639]
ddr    val     1864 [925  95 671  36 137]


100%|██████████| 516/516 [00:00<00:00, 192635.59it/s]


IDRiD 516 [168  25 168  93  62]
training images: 11520


## 5. Quantum circuit and model (identical to the multi-source notebook)

In [ ]:
n_qubits = 8
n_layers = 3

dev = qml.device("default.qubit", wires=n_qubits)

def quantum_circuit(inputs, weights):
    for i in range(n_qubits):                       # initial U3 encoding
        qml.U3(inputs[..., i], inputs[..., i], inputs[..., i], wires=i)
    for l in range(n_layers):
        for i in range(n_qubits):                   # data re-uploading
            qml.RY(inputs[..., i], wires=i)
        for i in range(n_qubits):                   # trainable single-qubit rotations (72 parameters in total)
            qml.U3(weights[l][i][0], weights[l][i][1], weights[l][i][2], wires=i)
        for i in range(n_qubits):                   # ring entanglement
            qml.CNOT(wires=[i, (i + 1) % n_qubits])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qnode = qml.QNode(quantum_circuit, dev, interface="torch", diff_method="backprop")

In [ ]:
class QuantumBranch(nn.Module):
    """256 -> attention gate -> reducer -> 8 angles -> (circuit | control) -> 8 features"""
    def __init__(self, feature_dim=256, mode="circuit"):
        super().__init__()
        self.mode = mode

        self.attn = nn.Sequential(
            nn.Linear(feature_dim, 64), nn.ReLU(), nn.Linear(64, feature_dim)
        )
        self.reducer = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),         nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),          nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, n_qubits)
        )

        if mode in ("circuit", "frozen"):
            self.quantum = qml.qnn.TorchLayer(qnode, weight_shapes)   # a fresh circuit for every model
            if mode == "frozen":                                      # random circuit that is never trained
                for p in self.quantum.parameters():
                    p.requires_grad_(False)
        elif mode == "mlp":                                           # matched classical layer: 8*8 + 8 = 72 parameters
            self.quantum = nn.Sequential(nn.Linear(n_qubits, n_qubits), nn.Tanh())
        elif mode == "skip":                                          # no circuit at all
            self.quantum = None
        else:
            raise ValueError(f"unknown quantum_mode: {mode}")

    def forward(self, x):
        a = torch.softmax(self.attn(x), dim=-1)
        x = x * a
        x = self.reducer(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = torch.tanh(x) * np.pi                       # 8 angles in (-pi, pi)

        if self.mode in ("circuit", "frozen"):
            q_out = self.quantum(x.float())
        elif self.mode == "mlp":
            q_out = self.quantum(x.float() / np.pi)
        else:                                            # "skip": same 8 numbers, scaled to (-1, 1) like the circuit output
            q_out = x.float() / np.pi
        return q_out, x

In [ ]:
class HybridMSLATNet(nn.Module):
    def __init__(self, num_classes=5, embed_dim=256, n_trans_layers=4, use_pos_emb=False,
                 quantum_mode="circuit", fixed_alpha=None):
        super().__init__()

        self.backbone = timm.create_model("efficientnet_b2", pretrained=True, features_only=True)
        feat_dims = [f["num_chs"] for f in self.backbone.feature_info[-3:]]

        self.proj = nn.ModuleList([nn.Linear(feat_dims[i], embed_dim) for i in range(3)])

        # optional learned position / scale embeddings (one vector per token of the concatenated sequence)
        self.pos_emb = None
        if use_pos_emb:
            was_training = self.backbone.training
            self.backbone.eval()
            with torch.no_grad():
                fs = self.backbone(torch.zeros(1, 3, 224, 224))[-3:]
            n_tokens = sum(f.shape[2] * f.shape[3] for f in fs)
            self.backbone.train(was_training)
            self.pos_emb = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
            nn.init.trunc_normal_(self.pos_emb, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=embed_dim * 4,
            dropout=0.5, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_trans_layers, enable_nested_tensor=False)

        self.pool = nn.Linear(embed_dim, 1)

        self.classical_head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

        self.quantum_branch = QuantumBranch(embed_dim, mode=quantum_mode)

        self.fusion_head = nn.Sequential(
            nn.Linear(embed_dim + n_qubits, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.quantum_head = nn.Sequential(
            nn.Linear(n_qubits, 32), nn.GELU(), nn.Linear(32, num_classes)
        )

        # mixing weight between the classical and the fused prediction
        self.learn_alpha = fixed_alpha is None
        if self.learn_alpha:
            self.alpha = nn.Parameter(torch.tensor(-1.0))               # sigmoid(-1) = 0.269 at the start
        else:
            self.register_buffer("alpha", torch.tensor(float(fixed_alpha)))

    def get_alpha(self):
        return torch.sigmoid(self.alpha) if self.learn_alpha else self.alpha

    def forward(self, x):
        feats = self.backbone(x)[-3:]

        tokens = []
        for i, f in enumerate(feats):
            t = f.flatten(2).transpose(1, 2)          # (B, H*W, C): every spatial position is one token
            tokens.append(self.proj[i](t))
        tokens = torch.cat(tokens, dim=1)
        if self.pos_emb is not None:
            tokens = tokens + self.pos_emb
        tokens = self.transformer(tokens)

        attn = torch.softmax(self.pool(tokens), dim=1)
        pooled = (tokens * attn).sum(dim=1)            # (B, 256)

        classical_logits = self.classical_head(pooled)

        quantum_features, q_input = self.quantum_branch(pooled)
        fusion_logits  = self.fusion_head(torch.cat([pooled, quantum_features], dim=1))
        quantum_logits = self.quantum_head(quantum_features)

        alpha = self.get_alpha()
        out = (1 - alpha) * classical_logits + alpha * fusion_logits

        return out, classical_logits, quantum_logits, pooled, attn, q_input

## 6. Shared helpers

In [ ]:
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
L5 = [0, 1, 2, 3, 4]

SPEC_TARGETS = (0.90, 0.80)

def sens_at_spec(Y, P, target_spec):
    """Highest referable-DR sensitivity at an operating point whose specificity is >= target_spec."""
    ref = (Y >= 2).astype(int)
    if ref.sum() in (0, len(ref)): return float("nan")
    fpr, tpr, _ = roc_curve(ref, P[:, 2:].sum(1))           # score = P(grade >= 2)
    ok = fpr <= 1 - target_spec + 1e-12
    return float(tpr[ok].max()) if ok.any() else 0.0

def metrics_from(P, Y):
    pred = P.argmax(1); cm = confusion_matrix(Y, pred, labels=L5)
    try:    auc = roc_auc_score(Y, P, multi_class="ovr", labels=L5)
    except ValueError: auc = float("nan")
    m = dict(acc=accuracy_score(Y, pred),
             f1=f1_score(Y, pred, average="macro", labels=L5, zero_division=0),
             qwk=cohen_kappa_score(Y, pred, weights="quadratic", labels=L5),
             auc=auc,
             ref_auc=roc_auc_score((Y >= 2).astype(int), P[:, 2:].sum(1)),
             sens=cm[2:, 2:].sum() / cm[2:].sum(), spec=cm[:2, :2].sum() / cm[:2].sum(),
             ref_tp=cm[2:, 2:].sum(), ref_fn=cm[2:, :2].sum(), ref_tn=cm[:2, :2].sum(), ref_fp=cm[:2, 2:].sum())
    for s in SPEC_TARGETS:
        m[f"sens@spec{int(round(s * 100))}"] = sens_at_spec(Y, P, s)
    rec = np.diag(cm) / np.maximum(cm.sum(1), 1)
    for g in L5:
        m[f"rec_{g}"] = rec[g]
    return {k: float(v) for k, v in m.items()}, cm

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for imgs, labels in loader:
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(imgs.to(device, non_blocking=True))[0]     # blended logits
        P.append(torch.softmax(out.float(), 1).cpu().numpy()); Y.append(labels.numpy())
    P = np.concatenate(P).astype(np.float64); P = P / P.sum(1, keepdims=True)
    Y = np.concatenate(Y)
    m, cm = metrics_from(P, Y)
    return m, P, Y, cm

def parse_run(run):
    """'v3_circuit_aNone_pe0_L4_s42' -> settings needed to rebuild the model"""
    mode = "circuit" if "_circuit_" in run else ("skip" if "_skip_" in run else None)
    a  = re.search(r"_a(None|[0-9.]+)_", run)
    pe = re.search(r"_pe(\d)_", run)
    L  = re.search(r"_L(\d+)_", run)
    s  = re.search(r"_s(\d+)$", run)
    return dict(mode=mode,
                fixed_alpha=None if (a is None or a.group(1) == "None") else float(a.group(1)),
                use_pos_emb=bool(pe and pe.group(1) == "1"),
                n_layers=int(L.group(1)) if L else 4,
                seed=int(s.group(1)) if s else None,
                multisource=run.startswith("ms_"))

def load_model(run, ckpt_path):
    c = parse_run(run)
    m = HybridMSLATNet(n_trans_layers=c["n_layers"], use_pos_emb=c["use_pos_emb"],
                       quantum_mode=c["mode"], fixed_alpha=c["fixed_alpha"]).to(device)
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    m.load_state_dict(sd, strict=True)
    be = ck.get("epoch", -1) + 1 if isinstance(ck, dict) else 0
    return m.eval(), be

## 7. Item 4: parameters per component and inference time

Random weights (timing does not depend on the trained values). GPU: batch 32 throughput and batch 1 latency, with the same mixed precision as evaluation. CPU: batch 1 latency in float32, the setting closest to a low-cost screening device. Median of repeated runs after a warm-up.

Skipped unless `RUN_TIMING = True` (the saved results from the earlier run are kept).

In [ ]:
if not RUN_TIMING:
    print("Timing skipped (RUN_TIMING = False); using the saved model_size_components.csv and inference_time.csv")
else:
    def count(mod): return sum(p.numel() for p in mod.parameters()) if mod is not None else 0

    def components(m):
        qb = m.quantum_branch
        vqc = count(qb.quantum) if qb.mode in ("circuit", "frozen") else 0
        return {"backbone (EfficientNet-B2)": count(m.backbone),
                "projections + Transformer + pooling": count(m.proj) + count(m.transformer) + count(m.pool)
                                                        + (m.pos_emb.numel() if m.pos_emb is not None else 0),
                "classical head": count(m.classical_head),
                "quantum branch, classical part": count(qb) - vqc,
                "VQC (trainable rotations)": vqc,
                "fusion head + quantum head + alpha": count(m.fusion_head) + count(m.quantum_head) + (1 if m.learn_alpha else 0),
                "TOTAL": count(m)}

    def time_model(m, dev, batch, n_warm, n_rep, amp):
        m = m.to(dev).eval(); x = torch.randn(batch, 3, 224, 224, device=dev); ts = []
        with torch.no_grad():
            for i in range(n_warm + n_rep):
                if dev.type == "cuda": torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.autocast(device_type=dev.type, enabled=amp):
                    m(x)
                if dev.type == "cuda": torch.cuda.synchronize()
                if i >= n_warm: ts.append(time.perf_counter() - t0)
        return float(np.median(ts))

    rows, comp = [], {}
    for mode in ("circuit", "skip"):
        torch.manual_seed(0)
        m = HybridMSLATNet(n_trans_layers=4, use_pos_emb=False, quantum_mode=mode, fixed_alpha=None)
        comp[mode] = components(m)
        r = {"model": "VQC (hybrid)" if mode == "circuit" else "no-VQC (skip control)",
             "params_M": comp[mode]["TOTAL"] / 1e6}
        if torch.cuda.is_available():
            g = torch.device("cuda")
            t32 = time_model(m, g, 32, 5, 20, True)
            r["GPU b32 ms/img"]   = 1000 * t32 / 32
            r["GPU b32 img/s"]    = 32 / t32
            r["GPU b1 ms"]        = 1000 * time_model(m, g, 1, 10, 50, True)
        m = m.cpu()
        r["CPU b1 ms"] = 1000 * time_model(m, torch.device("cpu"), 1, 3, 15, False)
        rows.append(r); del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print("Parameters per component")
    print(pd.DataFrame(comp).astype(int).to_string())
    tim = pd.DataFrame(rows)
    print("\nInference time | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
          "| CPU threads:", torch.get_num_threads())
    print(tim.round(2).to_string(index=False))
    pd.DataFrame(comp).to_csv(f"{BASE_PATH}/model_size_components.csv")
    tim.to_csv(f"{BASE_PATH}/inference_time.csv", index=False)
    print("\nsaved: model_size_components.csv, inference_time.csv")

Timing skipped (RUN_TIMING = False); using the saved model_size_components.csv and inference_time.csv


## 8. Item 5: AdaBN test-time adaptation

### 8a. Find the checkpoints
Lists every saved VQC / no-VQC best checkpoint (smoke runs excluded). **Check the printed list before going on:** you should see 3 APTOS-only VQC, 3 APTOS-only no-VQC, 3 multi-source VQC and 3 multi-source no-VQC runs. If one is missing (for example the APTOS-only VQC seed 43 trained in your other Google account), copy its `best_<RUN>.pth` into the `checkpoints` folder first.

In [ ]:
EXCLUDE = []      # run names to leave out (e.g. old debug runs), if the list below shows extras

found = []
for p in sorted(glob.glob(f"{BASE_PATH}/checkpoints/best_*.pth")):
    run = os.path.basename(p)[len("best_"):-len(".pth")]
    if "smoke" in run.lower() or run in EXCLUDE: continue
    c = parse_run(run)
    if c["mode"] is None: continue          # control families: added in a later version
    found.append(dict(run=run, path=p, family=("multi-source " if c["multisource"] else "APTOS-only ") +
                      ("VQC" if c["mode"] == "circuit" else "no-VQC"), seed=c["seed"]))
ck_df = pd.DataFrame(found)
print(ck_df[["family", "seed", "run"]].to_string(index=False))
print("\ncounts:", ck_df.family.value_counts().to_dict())

             family  seed                          run
   multi-source VQC    42  ms_circuit_aNone_pe0_L4_s42
   multi-source VQC    43  ms_circuit_aNone_pe0_L4_s43
   multi-source VQC    44  ms_circuit_aNone_pe0_L4_s44
multi-source no-VQC    42     ms_skip_aNone_pe0_L4_s42
multi-source no-VQC    43     ms_skip_aNone_pe0_L4_s43
multi-source no-VQC    44     ms_skip_aNone_pe0_L4_s44
  APTOS-only no-VQC    42 skip50_skip_aNone_pe0_L4_s42
  APTOS-only no-VQC    43 skip50_skip_aNone_pe0_L4_s43
  APTOS-only no-VQC    44 skip50_skip_aNone_pe0_L4_s44
     APTOS-only VQC    42  v3_circuit_aNone_pe0_L4_s42
     APTOS-only VQC    43  v3_circuit_aNone_pe0_L4_s43
     APTOS-only VQC    44  v3_circuit_aNone_pe0_L4_s44

counts: {'multi-source VQC': 3, 'multi-source no-VQC': 3, 'APTOS-only no-VQC': 3, 'APTOS-only VQC': 3}


### 8b. Target sets and the AdaBN step
`adabn` resets every batch-norm layer's running mean and variance and re-estimates them as a plain average over the adaptation images (no augmentation, dropout off). All other weights stay as trained.

In [ ]:
ddr_all = alldf[alldf.source == "ddr"]
TARGETS = {
    # name: (adaptation images, evaluation images)
    "idrid_full":    (idr, idr),
    "idrid_heldout": (idr[idr["sub"] == "train"], idr[idr["sub"] == "test"]),
    "ddr_full":      (ddr_all, ddr_all),
    "ddr_heldout":   (ddr_all[ddr_all.split == "train"], ddr_all[ddr_all.split == "test"]),
}
mk_loader = lambda df: DataLoader(CachedDS(df, val_tf), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
for k, (a, e) in TARGETS.items():
    print(f"{k:14s} adapt on {len(a):6d} | evaluate on {len(e):6d}", np.bincount(e.diagnosis, minlength=5))

BN = nn.modules.batchnorm._BatchNorm

@torch.no_grad()
def adabn(model, loader):
    model.eval()
    n_bn = 0
    for mod in model.modules():
        if isinstance(mod, BN):
            mod.reset_running_stats(); mod.momentum = None; mod.train(); n_bn += 1
    for imgs, _ in loader:
        with torch.autocast(device_type=device.type, enabled=use_amp):
            model(imgs.to(device, non_blocking=True))
    model.eval()
    return n_bn

idrid_full     adapt on    516 | evaluate on    516 [168  25 168  93  62]
idrid_heldout  adapt on    413 | evaluate on    103 [34  5 32 19 13]
ddr_full       adapt on  12424 | evaluate on  12424 [6168  630 4477  236  913]
ddr_heldout    adapt on   8696 | evaluate on   1864 [926  94 672  35 137]


### 8c. Run
For each checkpoint: (1) APTOS test without adaptation, as a load check (predictions saved); for multi-source models also DDR test as an internal held-out set; (2) for each external target, the unadapted result, then the adapted one (a fresh copy of the weights each time). Results go to `results_tta_v2.csv`; predictions to `tta_preds_v2/<RUN>__<target>__<condition>.npz`. Finished rows are skipped if the session breaks.

In [ ]:
tta_path = f"{BASE_PATH}/results_tta_v2.csv"       # new file: more columns than the old results_tta.csv
pred_dir = f"{BASE_PATH}/tta_preds_v2"; os.makedirs(pred_dir, exist_ok=True)
done = set()
if os.path.exists(tta_path):
    d = pd.read_csv(tta_path); done = set(zip(d.run, d.target, d.condition))
def log_tta(**row):
    pd.DataFrame([row]).to_csv(tta_path, mode="a", header=not os.path.exists(tta_path), index=False)

def ids_of(df):
    return (df["id_code"].astype(str) if "id_code" in df.columns
            else df["sub"].astype(str) + "_" + df["name"].astype(str)).to_numpy()

def save_preds(run, t, cond, P, Y, ids, cm):
    np.savez(f"{pred_dir}/{run}__{t}__{cond}.npz", P=P, Y=Y, ids=ids, cm=cm)

def show(t, cond, m):
    print(f"  {t:17s} {cond:6s} acc {m['acc']:.4f} | F1 {m['f1']:.4f} | QWK {m['qwk']:.4f} | "
          f"ref AUC {m['ref_auc']:.4f} | sens {m['sens']:.3f} spec {m['spec']:.3f} | sens@spec90 {m['sens@spec90']:.3f}")

# internal test sets (source only): same data frames, in the same order, as the test loaders
INTERNAL = {"aptos_test":        (alldf[(alldf.split == "test") & (alldf.source == "aptos")], test_loaders["aptos"]),
            "ddr_test_internal": (alldf[(alldf.split == "test") & (alldf.source == "ddr")],   test_loaders["ddr"])}

todo = ck_df.head(1) if SMOKE else ck_df
for _, r in todo.iterrows():
    run, ms = r.run, r.family.startswith("multi")
    internal = ["aptos_test"] + (["ddr_test_internal"] if ms else [])
    targets  = ["idrid_full", "idrid_heldout"] + ([] if (ms or SMOKE) else ["ddr_full", "ddr_heldout"])
    needed = [(run, t, "source") for t in internal] + [(run, t, c) for t in targets for c in ("source", "adabn")]
    if all(k in done for k in needed):
        print("SKIP (done):", run); continue
    t0 = time.time()
    try:
        model, be = load_model(run, r.path)
    except Exception as e:
        print(f"\n!!! LOAD FAILED for {run}: {str(e)[:300]}\n    (skipped; send me this message)"); continue

    # (1) internal test sets, no adaptation; APTOS test is also the load check
    apt_qwk = None
    for t in internal:
        e_df, e_loader = INTERNAL[t]
        m, P, Y, cm = evaluate(model, e_loader)
        if t == "aptos_test":
            apt_qwk = m["qwk"]
            print(f"\n=== {run} | {r.family} | best epoch {be} | APTOS test QWK {apt_qwk:.4f} (load check)")
        else:
            show(t, "source", m)
        if (run, t, "source") not in done:
            save_preds(run, t, "source", P, Y, ids_of(e_df), cm)
            log_tta(run=run, family=r.family, seed=r.seed, target=t, condition="source",
                    n_adapt=0, n_eval=len(Y), aptos_test_qwk=apt_qwk, **m)

    # (2) external targets: source, then AdaBN
    for t in targets:
        a_df, e_df = TARGETS[t]
        e_loader = mk_loader(e_df)
        for cond in ("source", "adabn"):
            if (run, t, cond) in done: continue
            model, _ = load_model(run, r.path)             # fresh weights every time
            if cond == "adabn": adabn(model, mk_loader(a_df))
            m, P, Y, cm = evaluate(model, e_loader)
            show(t, cond, m)
            save_preds(run, t, cond, P, Y, ids_of(e_df), cm)
            log_tta(run=run, family=r.family, seed=r.seed, target=t, condition=cond,
                    n_adapt=len(a_df) if cond == "adabn" else 0, n_eval=len(Y), aptos_test_qwk=apt_qwk, **m)
    del model
    if device.type == "cuda": torch.cuda.empty_cache()
    print(f"  ({(time.time() - t0) / 60:.1f} min)")
print("\nALL CHECKPOINTS IN THE LIST ARE FINISHED" + (" (SMOKE)" if SMOKE else ""))

SKIP (done): ms_circuit_aNone_pe0_L4_s42

=== ms_circuit_aNone_pe0_L4_s43 | multi-source VQC | best epoch 13 | APTOS test QWK 0.9014 (load check)
  ddr_test_internal source acc 0.8176 | F1 0.6324 | QWK 0.8476 | ref AUC 0.9558 | sens 0.876 spec 0.903 | sens@spec90 0.882
  idrid_full        source acc 0.4903 | F1 0.4524 | QWK 0.6970 | ref AUC 0.9344 | sens 0.895 spec 0.777 | sens@spec90 0.811
  idrid_full        adabn  acc 0.5698 | F1 0.4578 | QWK 0.6776 | ref AUC 0.9262 | sens 0.845 spec 0.834 | sens@spec90 0.783
  idrid_heldout     source acc 0.3981 | F1 0.3764 | QWK 0.5575 | ref AUC 0.8854 | sens 0.859 spec 0.641 | sens@spec90 0.719
  idrid_heldout     adabn  acc 0.4660 | F1 0.3815 | QWK 0.5210 | ref AUC 0.8630 | sens 0.844 spec 0.692 | sens@spec90 0.672
  (0.5 min)

=== ms_circuit_aNone_pe0_L4_s44 | multi-source VQC | best epoch 12 | APTOS test QWK 0.8977 (load check)
  ddr_test_internal source acc 0.8256 | F1 0.6303 | QWK 0.8440 | ref AUC 0.9544 | sens 0.878 spec 0.908 | sens@spec90

### 8d. Summary: mean ± SD over seeds, the AdaBN change, per-class recall and referable counts (all saved as CSV)

In [ ]:
t = pd.read_csv(tta_path)
KEYS = ["family", "target", "condition"]
cols = ["acc", "f1", "qwk", "auc", "ref_auc", "sens", "spec", "sens@spec90", "sens@spec80"]
pd.set_option("display.width", 250)

summ = t.groupby(KEYS)[cols].agg(["mean", "std"]).round(4)
summ[("n_seeds", "")] = t.groupby(KEYS)["seed"].nunique()
print("Mean and SD over seeds (SD is empty when there is only one seed)")
print(summ.to_string())
summ.to_csv(f"{BASE_PATH}/tta_summary_mean_sd.csv")

ext = t[t.target.isin(TARGETS.keys())]
wide = ext.pivot_table(index=["family", "seed", "target"], columns="condition", values=cols)
if "adabn" in wide.columns.get_level_values(1):
    delta = pd.DataFrame({c: wide[(c, "adabn")] - wide[(c, "source")] for c in cols}).dropna(how="all")
    print("\nAdapted minus source, per seed:")
    print(delta.round(4).to_string())
    dsum = delta.groupby(["family", "target"]).agg(["mean", "std"]).round(4)
    print("\nAdapted minus source, mean and SD over seeds:")
    print(dsum.to_string())
    delta.round(4).to_csv(f"{BASE_PATH}/tta_adabn_minus_source_per_seed.csv")
    dsum.to_csv(f"{BASE_PATH}/tta_adabn_minus_source_mean_sd.csv")

REC = [f"rec_{g}" for g in L5]
rec = t.groupby(KEYS)[REC].agg(["mean", "std"]).round(3)
print("\nPer-class recall (0 No DR, 1 Mild, 2 Moderate, 3 Severe, 4 Proliferative)")
print(rec.to_string())
rec.to_csv(f"{BASE_PATH}/tta_per_class_recall.csv")

cnt = t.groupby(KEYS)[["n_eval", "ref_tp", "ref_fn", "ref_tn", "ref_fp"]].mean().round(1)
print("\nReferable-DR counts, mean over seeds (TP = referable detected, FN = referable missed)")
print(cnt.to_string())
cnt.to_csv(f"{BASE_PATH}/tta_referable_counts.csv")
print("\nsaved: tta_summary_mean_sd.csv, tta_adabn_minus_source_*.csv, tta_per_class_recall.csv, tta_referable_counts.csv")

Mean and SD over seeds (SD is empty when there is only one seed)
                                                    acc              f1             qwk             auc         ref_auc            sens            spec         sens@spec90         sens@spec80         n_seeds
                                                   mean     std    mean     std    mean     std    mean     std    mean     std    mean     std    mean     std        mean     std        mean     std        
family              target            condition                                                                                                                                                                
APTOS-only VQC      aptos_test        source     0.8407  0.0059  0.6119  0.0090  0.8881  0.0123  0.9307  0.0158  0.9703  0.0138  0.9503  0.0365  0.9274  0.0185      0.9854  0.0051      0.9912  0.0088       3
                    ddr_full          adabn      0.6179  0.0199  0.4088  0.0121  0.6449  0.0260  0.7643

## 9. Paired bootstrap intervals

Each comparison resamples the same test images for both models and all shared seeds (1,000 resamples), and averages the difference over those seeds. The intervals reflect test-set sampling only, not variation between training runs. Comparisons whose prediction files do not exist yet (for example multi-source VQC seeds 43 and 44) are skipped, and the `n_seeds` column shows how many seeds went into each row.

In [ ]:
N_BOOT = 1000

def qwk_fast(y, pred):
    cm = np.bincount(y * 5 + pred, minlength=25).reshape(5, 5).astype(float)
    i, j = np.indices((5, 5)); W = (i - j) ** 2 / 16.0
    E = np.outer(cm.sum(1), cm.sum(0)) / cm.sum()
    den = (W * E).sum()
    return 1.0 - (W * cm).sum() / den if den > 0 else np.nan

FAST = {"qwk":     lambda y, P: qwk_fast(y, P.argmax(1)),
        "ref_auc": lambda y, P: (roc_auc_score((y >= 2).astype(int), P[:, 2:].sum(1))
                                 if 0 < (y >= 2).sum() < len(y) else np.nan)}

RUN_OF = {(f, int(s)): r for r, f, s in zip(ck_df.run, ck_df.family, ck_df.seed)}

def load_preds(family, seed, target, cond):
    z = np.load(f"{pred_dir}/{RUN_OF[(family, seed)]}__{target}__{cond}.npz", allow_pickle=True)
    return z["P"], z["Y"].astype(int), z["ids"]

def paired_bootstrap(fa, ca, fb, cb, target, metric, n_boot=N_BOOT, rs=0):
    fn = FAST[metric]
    seeds = sorted({s for f, s in RUN_OF if f == fa} & {s for f, s in RUN_OF if f == fb})
    pairs = []
    for s in seeds:
        try:
            Pa, y, ia = load_preds(fa, s, target, ca)
            Pb, yb, ib = load_preds(fb, s, target, cb)
        except (FileNotFoundError, KeyError):
            continue
        assert (ia == ib).all() and (y == yb).all(), f"image order differs: {fa} vs {fb}, {target}"
        pairs.append((Pa, Pb, y))
    if not pairs: return None
    n = len(pairs[0][2]); rng = np.random.default_rng(rs)
    obs = np.mean([fn(y, Pa) - fn(y, Pb) for Pa, Pb, y in pairs])
    boots = np.empty(n_boot)
    for b in range(n_boot):
        i = rng.integers(0, n, n)
        boots[b] = np.mean([fn(y[i], Pa[i]) - fn(y[i], Pb[i]) for Pa, Pb, y in pairs])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return dict(A=f"{fa} [{ca}]", B=f"{fb} [{cb}]", target=target, metric=metric, n_seeds=len(pairs),
                diff=round(float(obs), 4), ci_low=round(float(lo), 4), ci_high=round(float(hi), 4),
                excludes_0=bool(lo > 0 or hi < 0))

EXT = ["idrid_full", "idrid_heldout", "ddr_full", "ddr_heldout"]
COMPARISONS = (
    # does the circuit help? (same input, no-VQC control)
    [("APTOS-only VQC", "source", "APTOS-only no-VQC", "source", t) for t in ["aptos_test"] + EXT] +
    [("multi-source VQC", "source", "multi-source no-VQC", "source", t) for t in ["aptos_test", "ddr_test_internal", "idrid_full", "idrid_heldout"]] +
    # what does AdaBN change?
    [(f, "adabn", f, "source", t) for f in ("APTOS-only VQC", "APTOS-only no-VQC") for t in EXT] +
    [(f, "adabn", f, "source", t) for f in ("multi-source VQC", "multi-source no-VQC") for t in ["idrid_full", "idrid_heldout"]] +
    # does the circuit change how much AdaBN helps?
    [("APTOS-only VQC", "adabn", "APTOS-only no-VQC", "adabn", t) for t in EXT]
)

res = []
for fa, ca, fb, cb, tg in COMPARISONS:
    for met in ("qwk", "ref_auc"):
        r = paired_bootstrap(fa, ca, fb, cb, tg, met)
        if r is not None: res.append(r)
boot = pd.DataFrame(res)
print(boot.to_string(index=False))
boot.to_csv(f"{BASE_PATH}/tta_paired_bootstrap.csv", index=False)
print("\nsaved: tta_paired_bootstrap.csv  (diff = A minus B; excludes_0 = the 95% interval does not contain 0)")

                          A                            B            target  metric  n_seeds    diff  ci_low  ci_high  excludes_0
    APTOS-only VQC [source]   APTOS-only no-VQC [source]        aptos_test     qwk        3 -0.0053 -0.0269   0.0121       False
    APTOS-only VQC [source]   APTOS-only no-VQC [source]        aptos_test ref_auc        3  0.0034 -0.0103   0.0178       False
    APTOS-only VQC [source]   APTOS-only no-VQC [source]        idrid_full     qwk        3  0.0305  0.0004   0.0608        True
    APTOS-only VQC [source]   APTOS-only no-VQC [source]        idrid_full ref_auc        3  0.0024 -0.0099   0.0151       False
    APTOS-only VQC [source]   APTOS-only no-VQC [source]     idrid_heldout     qwk        3 -0.0175 -0.0878   0.0546       False
    APTOS-only VQC [source]   APTOS-only no-VQC [source]     idrid_heldout ref_auc        3  0.0007 -0.0344   0.0347       False
    APTOS-only VQC [source]   APTOS-only no-VQC [source]          ddr_full     qwk        3 -0.00

## 10. Package versions (for the implementation details in the paper)

In [ ]:
for p in ["torch", "torchvision", "timm", "pennylane", "albumentations", "numpy", "sklearn", "cv2", "pandas"]:
    try:    print(f"{p:15s}", getattr(importlib.import_module(p), "__version__", "?"))
    except ImportError: print(f"{p:15s} not installed")
print(f"{'python':15s}", sys.version.split()[0])
if torch.cuda.is_available(): print(f"{'GPU':15s}", torch.cuda.get_device_name(0))

torch           2.11.0+cu130
torchvision     0.26.0+cu130
timm            1.0.30
pennylane       0.45.1
albumentations  2.0.8
numpy           2.1.3
sklearn         1.6.1
cv2             4.14.0
pandas          2.2.3
python          3.13.15
GPU             Tesla T4
